# 🦙 Inferencia con Qwen 3.8 / GLM en Google Colab con Ollama

Este notebook está adaptado específicamente para ejecutar modelos de última generación y sin censura (**Abliterated**):
- 🚀 **Huihui-Qwen3.8-27B-abliterated-GGUF** (`Q4_K_M`)
- ⚡ **SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF** (`Q2_0` / `IQ3`)

### 🌟 Ventajas:
- Descarga directa de GGUFs desde Hugging Face mediante el conector nativo `hf.co/`.
- Expone un túnel público seguro con **Cloudflare Tunnel** compatible con la API de OpenAI.

## 1. Verificar GPU

In [ ]:
#@title 🔍 Comprobar GPU
!nvidia-smi

## 2. Instalar Ollama y Cloudflared

In [ ]:
#@title 📦 Instalar Ollama y Cloudflared
!curl -fsSL https://ollama.com/install.sh | sh

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
!pip install -q openai
print("✅ Ollama y Cloudflared instalados.")

## 3. Iniciar Servidor Ollama y Cloudflare Tunnel

In [ ]:
#@title 🚀 Iniciar Servicio Ollama y Túnel
import subprocess
import time
import re
import os

# Matar instancias previas si existen
!pkill -f ollama > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# Iniciar Ollama en segundo plano escuchando en todas las interfaces
ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "0.0.0.0:11434"
with open("ollama.log", "w") as ollama_log:
    subprocess.Popen("ollama serve", shell=True, env=ollama_env, stdout=ollama_log, stderr=ollama_log)

# Iniciar túnel de Cloudflare para el puerto 11434
with open("tunnel.log", "w") as tunnel_log:
    subprocess.Popen("cloudflared tunnel --url http://localhost:11434", shell=True, stdout=tunnel_log, stderr=tunnel_log)

print("⏳ Esperando que Ollama y Cloudflare inicien...")
public_url = None
for _ in range(25):
    time.sleep(2)
    try:
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print("=" * 65)
    print("🎉 ¡TÚNEL OLLAMA PÚBLICO DISPONIBLE!")
    print(f"🌐 Ollama Host / Base URL: {public_url}")
    print(f"🌐 OpenAI-compatible endpoint: {public_url}/v1")
    print("=" * 65)
else:
    print("⚠️ No se detectó la URL del túnel. Revisa tunnel.log")

## 4. Descargar Modelo Seleccionado desde Hugging Face

Ollama se conecta directamente a Hugging Face para descargar la cuantización GGUF elegida.

In [ ]:
#@title 📥 Seleccionar y Descargar Modelo
MODEL_CHOICE = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K_M)" #@param ["huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K_M)", "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF (Q2_0)", "personalizado"]
CUSTOM_TAG = "" #@param {type:"string"}

if MODEL_CHOICE == "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K_M)":
    MODEL_TAG = "hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M"
elif MODEL_CHOICE == "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF (Q2_0)":
    MODEL_TAG = "hf.co/SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF:Q2_0"
else:
    MODEL_TAG = CUSTOM_TAG.strip()

print(f"⏳ Descargando {MODEL_TAG} en la GPU de Colab...")
!ollama pull {MODEL_TAG}
print(f"\n✅ ¡Modelo {MODEL_TAG} listo para inferencia!")

## 5. Probar Inferencia con Cliente OpenAI

In [ ]:
#@title 🧪 Probar Endpoint con Python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = client.chat.completions.create(
    model=MODEL_TAG,
    messages=[
        {"role": "user", "content": "Escribe un hola mundo en Rust con comentarios explicativos."}
    ],
    stream=True
)

for chunk in response:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()